In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = "/Users/wangtong/Desktop/26 Spring/STAT 628/628m3-group6-main"
DATA_PATH = os.path.join(BASE_DIR, "processed_flights_utc.csv")
OUT_DIR = os.path.join(BASE_DIR, "preprocessing_figures")
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)

df["FlightDate"] = pd.to_datetime(df["FlightDate"])
df["CRSDepTime_UTC"] = pd.to_datetime(df["CRSDepTime_UTC"], utc=True)
df["CRSArrTime_UTC"] = pd.to_datetime(df["CRSArrTime_UTC"], utc=True)

sns.set_theme(style="whitegrid")


# =========================
# Figure 1: Pipeline overview counts
# =========================
summary = pd.DataFrame({
    "Step": [
        "Processed flight records",
        "Airlines retained",
        "Airports retained",
        "Origin-Destination routes"
    ],
    "Count": [
        len(df),
        df["Reporting_Airline"].nunique(),
        len(set(df["Origin"]).union(set(df["Dest"]))),
        df[["Origin", "Dest"]].drop_duplicates().shape[0]
    ]
})

plt.figure(figsize=(8, 4.8))
ax = sns.barplot(data=summary, x="Count", y="Step", color="#4C78A8")
for i, v in enumerate(summary["Count"]):
    ax.text(v, i, f" {v:,}", va="center", fontsize=10)
plt.title("Preprocessing Scope: From Monthly DOT Files to Modeling Dataset")
plt.xlabel("Count")
plt.ylabel("")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "preprocessing_scope_summary.png"), dpi=300)
plt.close()


# =========================
# Figure 2: Monthly coverage
# =========================
monthly = (
    df.groupby(["Year", "Month"])
      .size()
      .reset_index(name="n_flights")
)
monthly["Year-Month"] = monthly["Year"].astype(str) + "-" + monthly["Month"].astype(str).str.zfill(2)

plt.figure(figsize=(9, 4.8))
ax = sns.barplot(data=monthly, x="Year-Month", y="n_flights", color="#72B7B2")
ax.set_title("Final Dataset Coverage by Month")
ax.set_xlabel("")
ax.set_ylabel("Number of flights")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "monthly_coverage.png"), dpi=300)
plt.close()


# =========================
# Figure 3: Time-zone conversion / red-eye logic
# =========================
df["scheduled_duration_min"] = (
    df["CRSArrTime_UTC"] - df["CRSDepTime_UTC"]
).dt.total_seconds() / 60

df["is_redeye_like"] = (
    (df["CRSDepTime"].astype(str).str.zfill(4).str[:2].astype(int) >= 20) &
    (df["CRSArrTime"].astype(str).str.zfill(4).str[:2].astype(int) <= 8)
)

plot_df = df[df["scheduled_duration_min"].between(0, 600)].copy()

plt.figure(figsize=(8, 4.8))
sns.histplot(
    data=plot_df,
    x="scheduled_duration_min",
    hue="is_redeye_like",
    bins=40,
    element="step",
    stat="count",
    common_norm=False
)
plt.title("UTC Conversion Produces Valid Scheduled Flight Durations")
plt.xlabel("Scheduled duration after UTC conversion (minutes)")
plt.ylabel("Number of flights")
plt.legend(title="Red-eye-like flight", labels=["Yes", "No"])
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "utc_duration_check.png"), dpi=300)
plt.close()


# =========================
# Figure 4: Holiday feature
# =========================
holiday_summary = (
    df.groupby("days_to_holiday")
      .size()
      .reset_index(name="n_flights")
      .sort_values("days_to_holiday")
)

plt.figure(figsize=(8, 4.8))
sns.lineplot(data=holiday_summary, x="days_to_holiday", y="n_flights", marker="o")
plt.title("Engineered Holiday Feature: Distance to Nearest Major Holiday")
plt.xlabel("Days to nearest holiday")
plt.ylabel("Number of flights")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "holiday_feature_distribution.png"), dpi=300)
plt.close()


# =========================
# Figure 5: Airline-airport route heatmap
# =========================
route_airline = (
    df.groupby(["Reporting_Airline", "Origin"])
      .size()
      .reset_index(name="n_flights")
      .pivot(index="Reporting_Airline", columns="Origin", values="n_flights")
      .fillna(0)
)

plt.figure(figsize=(11, 3.8))
sns.heatmap(route_airline, cmap="Blues", linewidths=0.5)
plt.title("Retained Flight Volume by Airline and Origin Airport")
plt.xlabel("Origin airport")
plt.ylabel("Airline")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "airline_origin_heatmap.png"), dpi=300)
plt.close()

print(f"Saved preprocessing figures to: {OUT_DIR}")

Saved preprocessing figures to: /Users/wangtong/Desktop/26 Spring/STAT 628/628m3-group6-main/preprocessing_figures
